# Visualize CoolRun Route Alternatives

This notebook continues after notebook 05.

It generates scored OpenRouteService walking routes if needed, then creates an interactive Folium map.

It does not change route scoring logic.

In [ ]:
# Import the libraries we need.
# sys lets us add the project folder to Python's import path.
# os lets us check environment variables.
# Path helps us find files in outputs/.
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

In [ ]:
# Find the project root folder.
# If this notebook is opened from the notebooks/ folder, the project root is one level up.
# If it is opened from the project root, the current folder is already the project root.
current_dir = Path.cwd()

if (current_dir / ".env").exists():
    project_root = current_dir
else:
    project_root = current_dir.parent

# Add the project root to Python's import path.
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Load API keys and settings from .env.
load_dotenv(project_root / ".env", override=True)

In [ ]:
# Import reusable routing and visualization helpers.
# These are the same backend functions used by the FastAPI /score-routes endpoint.
from backend.app.routing.openroute_service import request_walking_routes
from backend.app.routing.route_scoring import score_route_features
from backend.app.visualization.route_maps import load_json, save_routes_map

In [ ]:
# These must match the Vienna orthofoto analysis settings used earlier.
metadata_path = project_root / "data" / "vienna_orthofoto_test_metadata.json"
if not metadata_path.exists():
    raise FileNotFoundError(
        f"Missing Vienna orthofoto metadata: {metadata_path}. Run notebooks/01b_vienna_orthofoto_test.ipynb first."
    )

metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
center_lat = metadata["center"]["lat"]
center_lon = metadata["center"]["lon"]
zoom = metadata["zoom"]
bbox = metadata["bbox"]
bbox_lonlat = metadata["bbox_lonlat"]
image_bbox = (bbox["min_x"], bbox["min_y"], bbox["max_x"], bbox["max_y"])

# Pick start and end points inside the current orthofoto bounds.
# Edit these coordinates for your demo route if you need a specific path.
start_lat = bbox_lonlat["south"] + 0.25 * (bbox_lonlat["north"] - bbox_lonlat["south"])
start_lon = bbox_lonlat["west"] + 0.20 * (bbox_lonlat["east"] - bbox_lonlat["west"])
end_lat = bbox_lonlat["south"] + 0.75 * (bbox_lonlat["north"] - bbox_lonlat["south"])
end_lon = bbox_lonlat["west"] + 0.80 * (bbox_lonlat["east"] - bbox_lonlat["west"])

print(f"Using Vienna orthofoto center: lat={center_lat}, lon={center_lon}")
print(f"Route start: lat={start_lat}, lon={start_lon}")
print(f"Route end: lat={end_lat}, lon={end_lon}")

# Keep this False for normal use.
# Set it to True if you change the start/end coordinates and want new routes.
force_regenerate_routes = False

In [ ]:
# Locate analysis outputs from the notebook workflow first.
# After notebook 05, detected trees and UTCI files usually live directly in outputs/.
# If those files are not there, this cell falls back to the newest FastAPI run folder.
outputs_dir = project_root / "outputs"
notebook_tree_geojson_path = outputs_dir / "detected_trees.geojson"

if notebook_tree_geojson_path.exists():
    run_dir = outputs_dir
    scored_routes_path = outputs_dir / "scored_routes.geojson"
    tree_geojson_path = notebook_tree_geojson_path
    utci_grid_path = outputs_dir / "utci_with_trees.npy"
    utci_summary_path = outputs_dir / "utci_summary.json"
    print("Using notebook outputs from outputs/.")
else:
    run_dirs = sorted(
        [path for path in outputs_dir.glob("run_*") if (path / "detected_trees.geojson").exists()],
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )

    if not run_dirs:
        raise FileNotFoundError(
            "No detected_trees.geojson found. Run notebooks 01-05 first, or run /analyze-area."
        )

    run_dir = run_dirs[0]
    scored_routes_path = run_dir / "scored_routes.geojson"
    tree_geojson_path = run_dir / "detected_trees.geojson"
    utci_grid_path = run_dir / "utci_with_trees.npy"
    utci_summary_path = run_dir / "utci_summary.json"
    print(f"Using FastAPI run folder: {run_dir}")

output_map_path = outputs_dir / "coolrun_routes_map.html"

print(f"Using detected trees: {tree_geojson_path}")
print(f"Route output path: {scored_routes_path}")

In [ ]:
# Generate scored_routes.geojson if it does not already exist.
# This calls OpenRouteService, so OPENROUTESERVICE_API_KEY must be set in .env.
if not os.getenv("OPENROUTESERVICE_API_KEY") and (force_regenerate_routes or not scored_routes_path.exists()):
    raise ValueError("OPENROUTESERVICE_API_KEY is missing. Add it to your .env file first.")

if force_regenerate_routes or not scored_routes_path.exists():
    print("No scored_routes.geojson found. Requesting 3 walking alternatives from OpenRouteService...")

    route_geojson = request_walking_routes(
        start_lon=start_lon,
        start_lat=start_lat,
        end_lon=end_lon,
        end_lat=end_lat,
        alternatives=3,
    )

    scored = score_route_features(
        route_features=route_geojson.get("features", []),
        tree_geojson_path=str(tree_geojson_path),
        utci_grid_path=str(utci_grid_path) if utci_grid_path.exists() else None,
        center_lon=center_lon,
        center_lat=center_lat,
        zoom=zoom,
        image_bbox=image_bbox,
    )

    output_features = []
    for route in scored["routes"]:
        output_features.append(
            {
                "type": "Feature",
                "geometry": route["geometry"],
                "properties": {
                    "id": route["id"],
                    "distance_m": route["distance_m"],
                    "sample_count": route["sample_count"],
                    "tree_density": route["tree_density"],
                    "average_utci": route["average_utci"],
                    "coolrun_score": route["coolrun_score"],
                },
            }
        )

    scored_routes = {
        "type": "FeatureCollection",
        "features": output_features,
        "selected": scored["selected"],
        "utci_available": scored["utci_available"],
    }

    scored_routes_path.write_text(json.dumps(scored_routes, indent=2), encoding="utf-8")
    print(f"Saved scored routes to: {scored_routes_path}")
else:
    print(f"Using existing scored routes: {scored_routes_path}")

In [ ]:
# Load the 3 route alternatives, detected trees, and optional UTCI summary.
scored_routes = load_json(str(scored_routes_path))
tree_geojson = load_json(str(tree_geojson_path))
utci_summary = load_json(str(utci_summary_path)) if utci_summary_path.exists() else None

print(f"Loaded routes: {len(scored_routes.get('features', []))}")
print(f"Loaded detected trees: {len(tree_geojson.get('features', []))}")
print(f"UTCI summary available: {utci_summary is not None}")
print(f"Selected routes: {scored_routes.get('selected')}")

In [ ]:
# Save an interactive Folium map with:
# - shortest route
# - greenest route
# - coolest route
# - detected tree points
# - start and end points
try:
    saved_map_path = save_routes_map(
        scored_routes=scored_routes,
        tree_geojson=tree_geojson,
        output_path=str(output_map_path),
        utci_summary=utci_summary,
    )
    print(f"Saved route map to: {saved_map_path}")
except ModuleNotFoundError as exc:
    if exc.name == "folium":
        print("Folium is not installed in this notebook kernel.")
        print("Run this in a notebook cell, then rerun this cell: %pip install folium")
    else:
        raise

In [ ]:
# Print route details so they are visible in the notebook output.
for feature in scored_routes.get("features", []):
    props = feature.get("properties", {})
    print("---")
    print(f"Route: {props.get('id')}")
    print(f"Distance: {props.get('distance_m', 0):.0f} m")
    print(f"Tree density: {props.get('tree_density', 0):.2f}")
    print(f"Average UTCI: {props.get('average_utci')}")
    print(f"CoolRun score: {props.get('coolrun_score')}")